In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("darkgrid")

### Dataset format

In [ ]:
DATASET_FOLDER = "../datasets/songdo_traffic/"

In [ ]:
df = pd.read_csv("{}/2022-10-04_A/2022-10-04_A_AM1.csv".format(DATASET_FOLDER))
df.head()

### Geographical coordinates transform

A giant orthophoto was built for the complete experiment region containing all 20 intersections, and these parameters are based on the complete orthophoto.

They are: longitude and latitude of the top-left corner of the complete orthophoto (point `O`), longitude latitude per pixel (so 4 numbers in total).

In [ ]:
ortho_params = np.loadtxt('{}/orthophotos/ortho_parameters.txt'.format(DATASET_FOLDER))
print(ortho_params)
lng_O, lat_O, lon_per_px, lat_per_px = ortho_params

As one can imagine, the complete orthophoto is too big to work with, so several cutouts were created (size 15000x15000 pixels) for each intersection.

In [ ]:
cutout_width_px = 15000 # original cutout size
width_half = cutout_width_px // 2 # 7500


These center files contain the pixel coordinates of the center point of the cutout in the complete orthophoto.

In [ ]:
A_center = np.loadtxt('{}/orthophotos/A_center.txt'.format(DATASET_FOLDER))
center_x, center_y = A_center
print(A_center)

The original sized cutouts were then resized for easier handling.

The `Ortho_X` and `Ortho_Y` columns in the dataset are based on the **resized** images.

In [ ]:
ortho_photo = plt.imread('{}/orthophotos/A.png'.format(DATASET_FOLDER))
resized_ortho_width_px = ortho_photo.shape[1] # e.g., 8000
print(ortho_photo.shape)


Therefore, we first obtain the longitude and latitude of the top-left corner of the original cutout (which will be the same as the top-left corner of the resized cutout).

Then we calculate the longitude and latitude of each point in the dataset by applying the pixel size scaling factor to the `Ortho_X` and `Ortho_Y` columns, and adding the offset from the top-left corner.

In [ ]:

lon_B = lng_O + (center_x - width_half) * lon_per_px
lat_B = lat_O + (center_y - width_half) * lat_per_px

scale = cutout_width_px / resized_ortho_width_px # e.g., 1.875
lon_per_px_resized = lon_per_px * scale
lat_per_px_resized = lat_per_px * scale

longitude = lon_B + df['Ortho_X'] * lon_per_px_resized
latitude = lat_B + df['Ortho_Y'] * lat_per_px_resized
assert np.allclose(longitude, df['Longitude'])
assert np.allclose(latitude, df['Latitude'])


Wrap everything and check again

In [ ]:
class OrthoCoordToGPSTransformer:

    def __init__(self, base_latitude, base_longitude, lon_per_px, lat_per_px):
        self.base_latitude = base_latitude
        self.base_longitude = base_longitude
        self.lon_per_px = lon_per_px
        self.lat_per_px = lat_per_px

    def __call__(self, ortho_x, ortho_y):
        longitude = self.base_longitude + ortho_x * self.lon_per_px
        latitude = self.base_latitude + ortho_y * self.lat_per_px
        return longitude, latitude

def get_ortho_to_gps_transformer(dataset_folder="../datasets/Sondo_Traffic/", intersection_label:str = 'A') -> callable:
    # the longitude, latitude of the top-left corner point O of the complete orthophoto (whole experiment area of 20 intersections)
    # the lngitude and latitude per pixel in the complete orthophoto (also for the original sized cutouts of 15000x15000 pixels)
    lng_O, lat_O, lon_per_px, lat_per_px = np.loadtxt('{}/orthophotos/ortho_parameters.txt'.format(dataset_folder))

    # center point x-y for the original sized cutout (15000x15000 pixels) in the complete orthophoto
    center_x, center_y = np.loadtxt('{}/orthophotos/{}_center.txt'.format(dataset_folder, intersection_label))
    cutout_width_px = 15000 # original cutout size, this is consistent
    width_half = cutout_width_px // 2 # 7500

    # the original sized cutout was resized for processing efficiency, so we need to calculate the scale factor
    ortho_photo = plt.imread('{}/orthophotos/{}.png'.format(dataset_folder, intersection_label))
    ortho_width_px = ortho_photo.shape[1] # e.g., 8000
    scale = cutout_width_px / ortho_width_px # e.g., 1.875

    # longitude-latitude of the top-left corner point B of the resized cutout
    lng_B = lng_O + (center_x - width_half) * lon_per_px
    lat_B = lat_O + (center_y - width_half) * lat_per_px

    lon_per_px_resized = lon_per_px * scale
    lat_per_px_resized = lat_per_px * scale

    ortho2gps = OrthoCoordToGPSTransformer(base_latitude=lat_B, base_longitude=lng_B, lon_per_px=lon_per_px_resized, lat_per_px=lat_per_px_resized)
    
    return ortho2gps

In [ ]:
ortho2gps_A = get_ortho_to_gps_transformer(dataset_folder=DATASET_FOLDER, intersection_label='A')
longitude, latitude = ortho2gps_A(df['Ortho_X'].to_numpy(), df['Ortho_Y'].to_numpy())
assert np.allclose(longitude, df['Longitude'])
assert np.allclose(latitude, df['Latitude'])

The local coordinates are given in KGD2002 / Central Belt 2010 planar coordinates (EPSG:5186)

In [ ]:
from pyproj import Transformer as ProjTransformer
# GPS coordinates are in WGS84 (EPSG:4326), and the local coordinates are in EPSG:5186 (KGD2002)
gps2local = ProjTransformer.from_crs("EPSG:4326", "EPSG:5186", always_xy=True)
x, y = gps2local.transform(longitude, latitude)
assert np.allclose(x, df['Local_X'])
assert np.allclose(y, df['Local_Y'])

### Transform Lane Coordinates

In [ ]:
lane_bbox_ortho_A = pd.read_csv("{}/segmentations/A.csv".format(DATASET_FOLDER))
lane_bbox_ortho_A.head()

In [ ]:
def transform_lane_bbox_ortho_to_local(lane_bbox_ortho, gps2local, ortho2gps):
    lane_bbox_local = pd.DataFrame(columns=lane_bbox_ortho.columns)
    lane_bbox_local['Section'] = lane_bbox_ortho['Section']
    lane_bbox_local['Lane'] = lane_bbox_ortho['Lane']

    # top-left, bottom-left, bottom-right, top-right
    point_labels = ['tl', 'bl', 'br', 'tr']
    for point_label in point_labels:
        ortho_x_col = '{}x'.format(point_label)
        ortho_y_col = '{}y'.format(point_label)
        longitude, latitude = ortho2gps(lane_bbox_ortho[ortho_x_col].to_numpy(), lane_bbox_ortho[ortho_y_col].to_numpy())
        local_x, local_y = gps2local.transform(longitude, latitude)
        lane_bbox_local['{}x'.format(point_label)] = local_x
        lane_bbox_local['{}y'.format(point_label)] = local_y
    
    return lane_bbox_local

In [ ]:
lane_bbox_local = transform_lane_bbox_ortho_to_local(lane_bbox_ortho_A, gps2local, ortho2gps_A)
lane_bbox_local.head()


In [ ]:
from pathlib import Path

folder = Path("{}/segmentations/".format(DATASET_FOLDER))
intersection_names = sorted([p.stem for p in folder.glob("*.csv")])

In [ ]:
lane_local_folder = Path("../datasets/songdo_drive/metadata/segmentations/")
lane_local_folder.mkdir(parents=True, exist_ok=True)

for intersection in intersection_names:
    print("Processing intersection {}...".format(intersection))
    lane_bbox_ortho = pd.read_csv("{}/segmentations/{}.csv".format(DATASET_FOLDER, intersection))
    ortho2gps = get_ortho_to_gps_transformer(dataset_folder=DATASET_FOLDER, intersection_label=intersection)
    lane_bbox_local = transform_lane_bbox_ortho_to_local(lane_bbox_ortho, gps2local, ortho2gps)
    output_path = Path("{}/{}_local.csv".format(lane_local_folder, intersection))
    if output_path.exists():
        print("Warning: {} already exists, skip overwriting.".format(output_path))
    else:
        lane_bbox_local.to_csv(output_path, index=False)

### Visualize and check the results

In [ ]:
# Configure visualization target
date_str = "2022-10-04"
intersection = "K"
frame_step = 25  # animate per 25 frame (1s)
max_frames = 60  # 1 min duration, gifs can't be too big

session_folder = Path(f"{DATASET_FOLDER}/{date_str}_{intersection}")
session_candidates = sorted(session_folder.glob(f"{date_str}_{intersection}_*.csv"))
trajectory_csv = session_candidates[4]  # usually AM5
lane_csv = Path(f"{lane_local_folder}/{intersection}_local.csv")

print(f"Using trajectory file: {trajectory_csv.name}")
print(f"Using lane file: {lane_csv.name}")

Display the ortho photo, so that one can check the lane layout

In [ ]:
ortho_photo = plt.imread('{}/orthophotos/{}.png'.format(DATASET_FOLDER, intersection))
plt.imshow(ortho_photo)

In [ ]:
from pathlib import Path
from matplotlib.animation import FuncAnimation

traj = pd.read_csv(
    trajectory_csv,
    usecols=["Vehicle_ID", "Local_Time", "Local_X", "Local_Y"],
)
lane_boxes = pd.read_csv(lane_csv)

traj = traj.dropna(subset=["Local_X", "Local_Y", "Local_Time"]).copy()
traj["time_td"] = pd.to_timedelta(traj["Local_Time"])
traj = traj.sort_values(["time_td", "Vehicle_ID"])

frame_table = (
    traj[["time_td", "Local_Time"]]
    .drop_duplicates()
    .sort_values("time_td")
    .reset_index(drop=True)
)

frame_table = frame_table.iloc[::frame_step].reset_index(drop=True)
if max_frames is not None:
    frame_table = frame_table.iloc[:max_frames].reset_index(drop=True)

frame_times = frame_table["time_td"].tolist()
frame_labels = frame_table["Local_Time"].tolist()

points_by_time = {
    t: group[["Local_X", "Local_Y"]].to_numpy()
    for t, group in traj.groupby("time_td", sort=True)
}

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Local_X (m)")
ax.set_ylabel("Local_Y (m)")
ax.set_title(f"Intersection {intersection} ({date_str}): Vehicle trajectories and lane boxes")
ax.grid(True, linewidth=0.4, alpha=0.4)

lane_points = lane_boxes[["tlx", "tly", "blx", "bly", "brx", "bry", "trx", "try"]].to_numpy()
for pts in lane_points:
    polygon = np.array(
        [
            [pts[0], pts[1]],
            [pts[2], pts[3]],
            [pts[4], pts[5]],
            [pts[6], pts[7]],
            [pts[0], pts[1]],
        ]
    )
    ax.plot(polygon[:, 0], polygon[:, 1], color="black", linewidth=0.7, alpha=0.6)

lane_x = lane_boxes[["tlx", "blx", "brx", "trx"]].to_numpy().ravel()
lane_y = lane_boxes[["tly", "bly", "bry", "try"]].to_numpy().ravel()
traj_x = traj["Local_X"].to_numpy()
traj_y = traj["Local_Y"].to_numpy()

x_min = min(lane_x.min(), traj_x.min())
x_max = max(lane_x.max(), traj_x.max())
y_min = min(lane_y.min(), traj_y.min())
y_max = max(lane_y.max(), traj_y.max())
padding = 10.0
ax.set_xlim(x_min - padding, x_max + padding)
ax.set_ylim(y_min - padding, y_max + padding)

scatter = ax.scatter([], [], s=8, c="#1f77b4", alpha=0.75)
time_text = ax.text(
    0.01,
    0.99,
    "",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"),
)


def init():
    scatter.set_offsets(np.empty((0, 2)))
    time_text.set_text("")
    return scatter, time_text


def update(frame_idx):
    t = frame_times[frame_idx]
    points = points_by_time.get(t, np.empty((0, 2)))
    scatter.set_offsets(points if len(points) else np.empty((0, 2)))
    time_text.set_text(f"Local_Time: {frame_labels[frame_idx]} | Vehicles: {len(points)}")
    return scatter, time_text


fps = 15
ani = FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=len(frame_times),
    interval=max(1, int(round(1000 / fps))),
    blit=True,
)

plt.close(fig)
ani.save(f"intersection_{intersection}_{date_str}.gif", writer="pillow", fps=fps)
print(f"Animation saved as intersection_{intersection}_{date_str}.gif")


Display animation, this can be slow if on a remote server, so we save the animation as a gif file for download.

In [ ]:
# from IPython.display import HTML
# HTML(ani.to_jshtml())